<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week9_Lecture_examples_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# INFO 212: Data Science Programming 1
___

### Week 9: Data Analysis Examples

---

**Question:**
- What can I learn from real world data analysis examples? 

**Objectives:**
- Apply the techiques learned in this course to real world data analysis problems

## US Baby Names 1880–2010
The United States Social Security Administration (SSA) has made available data on
the frequency of baby names from 1880 to 2010. 

There are many things you might want to do with the dataset:
* Visualize the proportion of babies given a particular name (your own, or another name) over time
* Determine the relative rank of a name
* Determine the most popular names in each year or the names whose popularity has advanced or declined the most
* Analyze trends in names: vowels, consonants, length, overall diversity, changes in spelling, first and last letters
* Analyze external sources of trends: biblical names, celebrities, demographic changes

In [ ]:
!head -n 10 datasets/babynames/yob1880.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
names1880 = pd.read_csv('datasets/babynames/yob1880.txt',
                        names=['name', 'sex', 'births'])
names1880

In [ ]:
names1880.groupby('sex').births.sum()

Since the dataset is split into files by year, one of the first things to do is to assemble
all of the data into a single DataFrame and further to add a year field. You can do this
using pandas.concat:

In [ ]:
years = range(1880, 2011)

pieces = []

columns = ['name', 'sex', 'births']

for year in years:
    path = 'datasets/babynames/yob%d.txt' % year
    frame = pd.read_csv(path, names=columns)

    frame['year'] = year
    pieces.append(frame)

    

In [ ]:
len(pieces)

In [ ]:
pieces[4].head()

In [ ]:
# Concatenate everything into a single DataFrame
names = pd.concat(pieces, ignore_index=True)

In [ ]:
names

In [ ]:
total_births = names.pivot_table('births', index='year',
                                 columns='sex', aggfunc=sum)

total_births.head()

In [ ]:
total_births.plot(title='Total births by sex and year')

Next, let’s insert a column prop with the fraction of babies given each name relative to
the total number of births. A prop value of 0.02 would indicate that 2 out of every
100 babies were given a particular name. Thus, we group the data by year and sex,
then add the new column to each group:

In [ ]:
def add_prop(group):
    group['prop'] = group.births / group.births.sum()
    return group

In [ ]:
names = names.groupby(['year', 'sex']).apply(add_prop)

In [ ]:
names

In [ ]:
names.groupby(['year', 'sex']).prop.sum()

Now that this is done, I’m going to extract a subset of the data to facilitate further
analysis: the top 1,000 names for each sex/year combination. This is yet another
group operation:

In [ ]:
def get_top1000(group):
    return group.sort_values(by='births', ascending=False)[:1000]

In [ ]:
grouped = names.groupby(['year', 'sex'])

top1000 = grouped.apply(get_top1000)

In [ ]:
top1000

In [ ]:
top1000.loc[1880, 'F']

In [ ]:
# Drop the group index, not needed
top1000.reset_index(inplace=True, drop=True)

In [ ]:
top1000

### Analyzing Naming Trends
With the full dataset and Top 1,000 dataset in hand, we can start analyzing various
naming trends of interest. Splitting the Top 1,000 names into the boy and girl portions
is easy to do first:

In [ ]:
boys = top1000[top1000.sex == 'M']
girls = top1000[top1000.sex == 'F']

In [ ]:
total_births = top1000.pivot_table('births', index='year',
                                   columns='name',
                                   aggfunc=sum)

In [ ]:
total_births.info()

In [ ]:
total_births.head()

In [ ]:
subset = total_births[['John', 'Jacob', 'Harry', 'Mary', 'Marilyn', 'Aaron']]

In [ ]:
subset.plot(subplots=True, figsize=(12, 10), grid=False,
            title="Number of births per year")

#### Measuring the increase in naming diversity
One explanation for the decrease in plots is that fewer parents are choosing common
names for their children. This hypothesis can be explored and confirmed in the data.
One measure is the proportion of births represented by the top 1,000 most popular
names.

In [ ]:
plt.figure()

In [ ]:
table = top1000.pivot_table('prop', index='year',
                            columns='sex', aggfunc=sum)

In [ ]:
table.plot(title='Sum of table1000.prop by year and sex',
           yticks=np.linspace(0, 1.2, 13), xticks=range(1880, 2020, 10))

You can see that, indeed, there appears to be increasing name diversity (decreasing
total proportion in the top 1,000). Another interesting metric is the number of distinct
names, taken in order of popularity from highest to lowest, in the top 50% of
births. This number is a bit more tricky to compute. Let’s consider just the boy names
from 2010:

In [ ]:
df = boys[boys.year == 2010]
df

After sorting prop in descending order, we want to know how many of the most popular
names it takes to reach 50%. You could write a for loop to do this, but a vectorized
NumPy way is a bit more clever. Taking the cumulative sum, cumsum, of prop and
then calling the method searchsorted returns the position in the cumulative sum at
which 0.5 would need to be inserted to keep it in sorted order:

In [ ]:
prop_cumsum = df.sort_values(by='prop', ascending=False).prop.cumsum()
prop_cumsum[:10]

In [ ]:
prop_cumsum.values.searchsorted(0.5)

Since arrays are zero-indexed, adding 1 to this result gives you a result of 117. By contrast,
in 1900 this number was much smaller:

In [ ]:
df = boys[boys.year == 1900]
in1900 = df.sort_values(by='prop', ascending=False).prop.cumsum()
in1900.values.searchsorted(0.5) + 1

You can now apply this operation to each year/sex combination, groupby those fields,
and apply a function returning the count for each group:

In [ ]:
def get_quantile_count(group, q=0.5):
    group = group.sort_values(by='prop', ascending=False)
    return group.prop.cumsum().values.searchsorted(q) + 1

In [ ]:
diversity = top1000.groupby(['year', 'sex']).apply(get_quantile_count)
diversity.head()

In [ ]:
total_births.head()

In [ ]:
diversity = diversity.unstack('sex')

In [ ]:
diversity

In [ ]:
fig = plt.figure()

In [ ]:
diversity.head()
diversity.plot(title="Number of popular names in top 50%")

#### The “last letter” revolution
In 2007, baby name researcher Laura Wattenberg pointed out on her website that the
distribution of boy names by final letter has changed significantly over the last 100
years. To see this, we first aggregate all of the births in the full dataset by year, sex, and
final letter:

In [ ]:
# extract last letter from name column
get_last_letter = lambda x: x[-1]
last_letters = names.name.map(get_last_letter)
last_letters

In [ ]:
last_letters.name = 'last_letter'

table = names.pivot_table('births', index=last_letters,
                          columns=['sex', 'year'], aggfunc=sum)

In [ ]:
table

Then we select out three representative years spanning the history and print the first
few rows:

In [ ]:
subtable = table.reindex(columns=[1910, 1960, 2010], level='year')
subtable

Next, normalize the table by total births to compute a new table containing proportion
of total births for each sex ending in each letter:

In [ ]:
subtable.sum()

In [ ]:
letter_prop = subtable / subtable.sum()
letter_prop

With the letter proportions now in hand, we can make bar plots for each sex broken
down by year

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 8))
letter_prop['M'].plot(kind='bar', rot=0, ax=axes[0], title='Male')
letter_prop['F'].plot(kind='bar', rot=0, ax=axes[1], title='Female',
                      legend=False)
plt.subplots_adjust(hspace=0.25)

As you can see, boy names ending in n have experienced significant growth since the
1960s.

In [ ]:
letter_prop = table / table.sum()
dny_ts = letter_prop.loc[['d', 'n', 'y'], 'M'].T
dny_ts.head()

With this DataFrame of time series in hand, I can make a plot of the trends over time
again with its plot method

In [ ]:
plt.close('all')

In [ ]:
fig = plt.figure()

In [ ]:
dny_ts.plot()

#### Boy names that became girl names (and vice versa)
Another fun trend is looking at boy names that were more popular with one sex earlier
in the sample but have “changed sexes” in the present. One example is the name
Lesley or Leslie. Going back to the top1000 DataFrame, I compute a list of names
occurring in the dataset starting with “lesl”:

In [ ]:
all_names = pd.Series(top1000.name.unique())

In [ ]:
all_names

In [ ]:
lesley_like = all_names[all_names.str.lower().str.contains('lesl')]
lesley_like

From there, we can filter down to just those names and sum births grouped by name
to see the relative frequencies:

In [ ]:
filtered = top1000[top1000.name.isin(lesley_like)]

In [ ]:
filtered.groupby('name').births.sum()

Next, let’s aggregate by sex and year and normalize within year:

In [ ]:
table = filtered.pivot_table('births', index='year',
                             columns='sex', aggfunc='sum')

In [ ]:
table.head()

In [ ]:
table = table.div(table.sum(1), axis=0)
table.tail()

In [ ]:
table.sample(10)

In [ ]:
fig = plt.figure()

In [ ]:
table.plot(style={'M': 'k-', 'F': 'k--'})

## MovieLens 1M Dataset
GroupLens Research provides a number of collections of movie ratings data collected
from users of MovieLens in the late 1990s and early 2000s. The data provide movie
ratings, movie metadata (genres and year), and demographic data about the users
(age, zip code, gender identification, and occupation). Such data is often of interest in
the development of recommendation systems based on machine learning algorithms.
While we do not explore machine learning techniques in detail in this book, I will
show you how to slice and dice datasets like these into the exact form you need.
The MovieLens 1M dataset contains 1 million ratings collected from 6,000 users on
4,000 movies. It’s spread across three tables: ratings, user information, and movie
information. After extracting the data from the ZIP file, we can load each table into a
pandas DataFrame object using pandas.read_table:

In [ ]:
import pandas as pd

# Make display smaller
pd.options.display.max_rows = 10

unames = ['user_id', 'gender', 'age', 'occupation', 'zip']
users = pd.read_table('datasets/movielens/users.dat', sep='::',
                      header=None, names=unames)

rnames = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_table('datasets/movielens/ratings.dat', sep='::',
                        header=None, names=rnames)

mnames = ['movie_id', 'title', 'genres']
movies = pd.read_table('datasets/movielens/movies.dat', sep='::',
                       header=None, names=mnames)

In [ ]:
users.head()

In [ ]:
ratings.head()

In [ ]:
movies[:5]

In [ ]:
movies[movies.movie_id == 1193]

In [ ]:
data = pd.merge(pd.merge(ratings, users), movies)
data

In [ ]:
data.iloc[0]

To get mean movie ratings for each film grouped by gender, we can use the
pivot_table method:

In [ ]:
mean_ratings = data.pivot_table('rating', index='title',
                                columns='gender', aggfunc='mean')
mean_ratings.head()

In [ ]:
ratings_by_title = data.groupby('title').size()

ratings_by_title[:10]

In [ ]:
active_titles = ratings_by_title.index[ratings_by_title >= 250]
active_titles

The index of titles receiving at least 250 ratings can then be used to select rows from
mean_ratings:

In [ ]:
# Select rows on the index
mean_ratings = mean_ratings.loc[active_titles]
mean_ratings

In [ ]:
top_female_ratings = mean_ratings.sort_values(by='F', ascending=False)
top_female_ratings[:10]

### Measuring Rating Disagreement

In [ ]:
mean_ratings['diff'] = mean_ratings['M'] - mean_ratings['F']

In [ ]:
sorted_by_diff = mean_ratings.sort_values(by='diff')
sorted_by_diff[:10]

In [ ]:
# Reverse order of rows, take first 10 rows
sorted_by_diff[::-1][:10]

Suppose instead you wanted the movies that elicited the most disagreement among
viewers, independent of gender identification. Disagreement can be measured by the
variance or standard deviation of the ratings:

In [ ]:
# Standard deviation of rating grouped by title
rating_std_by_title = data.groupby('title')['rating'].std()

In [ ]:
rating_std_by_title

In [ ]:
# Filter down to active_titles
rating_std_by_title = rating_std_by_title.loc[active_titles]

In [ ]:
# Order Series by value in descending order
rating_std_by_title.sort_values(ascending=False)[-10:]

## 2012 Federal Election Commission Database
The US Federal Election Commission publishes data on contributions to political
campaigns. This includes contributor names, occupation and employer, address, and
contribution amount. An interesting dataset is from the 2012 US presidential election.
A version of the dataset is a 150 megabyte CSV file
P00000001-ALL.csv

```
fec = pd.read_csv('datasets/fec/P00000001-ALL.csv')
fec.info()
```

In [ ]:
fec = pd.read_csv('datasets/fec/P00000001-ALL.csv')
fec.info()

A simple records looks like:

```
fec.iloc[123456]
```

In [ ]:
fec.iloc[123456]

You may think of some ways to start slicing and dicing this data to extract informative
statistics about donors and patterns in the campaign contributions. Here is 
a number of different analyses that apply techniques in this course.

You can see that there are no political party affiliations in the data, so this would be
useful to add. You can get a list of all the unique political candidates using unique:

```
unique_cands = fec.cand_nm.unique()
unique_cands
```

In [ ]:
unique_cands = fec.cand_nm.unique()
unique_cands

One way to indicate party affiliation is using a dict (for illustration purpose only):

```
parties = {'Bachmann, Michelle': 'Republican',
           'Cain, Herman': 'Republican',
           'Gingrich, Newt': 'Republican',
           'Huntsman, Jon': 'Republican',
           'Johnson, Gary Earl': 'Republican',
           'McCotter, Thaddeus G': 'Republican',
           'Obama, Barack': 'Democrat',
           'Paul, Ron': 'Republican',
           'Pawlenty, Timothy': 'Republican',
           'Perry, Rick': 'Republican',
           "Roemer, Charles E. 'Buddy' III": 'Republican',
           'Romney, Mitt': 'Republican',
           'Santorum, Rick': 'Republican'}
```

In [ ]:
parties = {'Bachmann, Michelle': 'Republican',
           'Cain, Herman': 'Republican',
           'Gingrich, Newt': 'Republican',
           'Huntsman, Jon': 'Republican',
           'Johnson, Gary Earl': 'Republican',
           'McCotter, Thaddeus G': 'Republican',
           'Obama, Barack': 'Democrat',
           'Paul, Ron': 'Republican',
           'Pawlenty, Timothy': 'Republican',
           'Perry, Rick': 'Republican',
           "Roemer, Charles E. 'Buddy' III": 'Republican',
           'Romney, Mitt': 'Republican',
           'Santorum, Rick': 'Republican'}

Now, using this mapping and the map method on Series objects, you can compute an
array of political parties from the candidate names:

```
fec.cand_nm[123456:123461]
```

In [ ]:
fec.cand_nm[123456:123461]

```
fec.cand_nm[123456:123461].map(parties)
```

In [ ]:
fec.cand_nm[123456:123461].map(parties)

```
fec['party'] = fec.cand_nm.map(parties)
fec['party'].value_counts()
```

In [ ]:
fec['party'] = fec.cand_nm.map(parties)
fec['party'].value_counts()

A couple of data preparation points. First, this data includes both contributions and
refunds (negative contribution amount):

```
(fec.contb_receipt_amt > 0).value_counts()
```

In [ ]:
(fec.contb_receipt_amt > 0).value_counts()

To simplify the analysis, I’ll restrict the dataset to positive contributions:

```
fec = fec[fec.contb_receipt_amt > 0]
```

In [ ]:
fec = fec[fec.contb_receipt_amt > 0]

Since Barack Obama and Mitt Romney were the main two candidates, I’ll also prepare
a subset that just has contributions to their campaigns:

```
fec_mrbo = fec[fec.cand_nm.isin(['Obama, Barack', 'Romney, Mitt'])]
```

In [ ]:
fec_mrbo = fec[fec.cand_nm.isin(['Obama, Barack', 'Romney, Mitt'])]

### Donation Statistics by Occupation and Employer
Donations by occupation is another oft-studied statistic. For example, lawyers (attorneys)
tend to donate more money to Democrats, while business executives tend to
donate more to Republicans. You have no reason to believe it; you can see for yourself
in the data. First, the total number of donations by occupation is easy:

```
fec.contbr_occupation.value_counts()[:10]
```

In [ ]:
fec.contbr_occupation.value_counts()[:10]

You will notice by looking at the occupations that many refer to the same basic job
type, or there are several variants of the same thing. The following code snippet illustrates
a technique for cleaning up a few of them by mapping from one occupation to
another; note the “trick” of using dict.get to allow occupations with no mapping to
“pass through”:

```
occ_mapping = {
   'INFORMATION REQUESTED PER BEST EFFORTS' : 'NOT PROVIDED',
   'INFORMATION REQUESTED' : 'NOT PROVIDED',
   'INFORMATION REQUESTED (BEST EFFORTS)' : 'NOT PROVIDED',
   'C.E.O.': 'CEO'
}
```

In [ ]:
occ_mapping = {
   'INFORMATION REQUESTED PER BEST EFFORTS' : 'NOT PROVIDED',
   'INFORMATION REQUESTED' : 'NOT PROVIDED',
   'INFORMATION REQUESTED (BEST EFFORTS)' : 'NOT PROVIDED',
   'C.E.O.': 'CEO'
}

```
f = lambda x: occ_mapping.get(x, x)
fec.contbr_occupation = fec.contbr_occupation.map(f)
```

In [ ]:
f = lambda x: occ_mapping.get(x, x)
fec.contbr_occupation = fec.contbr_occupation.map(f)

Do the same thing for employer:

```
emp_mapping = {
   'INFORMATION REQUESTED PER BEST EFFORTS' : 'NOT PROVIDED',
   'INFORMATION REQUESTED' : 'NOT PROVIDED',
   'SELF' : 'SELF-EMPLOYED',
   'SELF EMPLOYED' : 'SELF-EMPLOYED',
}

# If no mapping provided, return x
f = lambda x: emp_mapping.get(x, x)
fec.contbr_employer = fec.contbr_employer.map(f)
```

In [ ]:
emp_mapping = {
   'INFORMATION REQUESTED PER BEST EFFORTS' : 'NOT PROVIDED',
   'INFORMATION REQUESTED' : 'NOT PROVIDED',
   'SELF' : 'SELF-EMPLOYED',
   'SELF EMPLOYED' : 'SELF-EMPLOYED',
}

# If no mapping provided, return x
f = lambda x: emp_mapping.get(x, x)
fec.contbr_employer = fec.contbr_employer.map(f)

Now, you can use pivot_table to aggregate the data by party and occupation, then
filter down to the subset that donated at least $2 million overall:

```
by_occupation = fec.pivot_table('contb_receipt_amt',
                                index='contbr_occupation',
                                columns='party', aggfunc='sum')
```

In [ ]:
by_occupation = fec.pivot_table('contb_receipt_amt',
                                index='contbr_occupation',
                                columns='party', aggfunc='sum')

```
over_2mm = by_occupation[by_occupation.sum(1) > 2000000]
over_2mm
```

In [ ]:
over_2mm = by_occupation[by_occupation.sum(1) > 2000000]
over_2mm

It can be easier to look at this data graphically as a bar plot ('barh' means horizontal
bar plot:

```
plt.figure()
```

In [ ]:
plt.figure()

```
over_2mm.plot(kind='barh')
```

In [ ]:
over_2mm.plot(kind='barh')

You might be interested in the top donor occupations or top companies that donated
to Obama and Romney. To do this, you can group by candidate name and use a variant
of the top method from earlier in the chapter:

```
def get_top_amounts(group, key, n=5):
    totals = group.groupby(key)['contb_receipt_amt'].sum()
    return totals.nlargest(n)
```

In [ ]:
def get_top_amounts(group, key, n=5):
    totals = group.groupby(key)['contb_receipt_amt'].sum()
    return totals.nlargest(n)

Then aggregate by occupation and employer:

```
grouped = fec_mrbo.groupby('cand_nm')
```

In [ ]:
grouped = fec_mrbo.groupby('cand_nm')

```
grouped.apply(get_top_amounts, 'contbr_employer', n=10)
```

In [ ]:
grouped.apply(get_top_amounts, 'contbr_employer', n=10)

```
grouped.apply(get_top_amounts, 'contbr_employer', n=10)
```

In [ ]:
grouped.apply(get_top_amounts, 'contbr_employer', n=10)

### Bucketing Donation Amounts
A useful way to analyze this data is to use the cut function to discretize the contributor
amounts into buckets by contribution size:

```
bins = np.array([0, 1, 10, 100, 1000, 10000,
                 100000, 1000000, 10000000])
labels = pd.cut(fec_mrbo.contb_receipt_amt, bins)
labels

```

In [ ]:
bins = np.array([0, 1, 10, 100, 1000, 10000,
                 100000, 1000000, 10000000])
labels = pd.cut(fec_mrbo.contb_receipt_amt, bins)
labels

We can then group the data for Obama and Romney by name and bin label to get a
histogram by donation size:

```
grouped = fec_mrbo.groupby(['cand_nm', labels])
grouped.size().unstack(0)
```

In [ ]:
grouped = fec_mrbo.groupby(['cand_nm', labels])
grouped.size().unstack(0)

This data shows that Obama received a significantly larger number of small donations
than Romney. You can also sum the contribution amounts and normalize within
buckets to visualize percentage of total donations of each size by candidate:

```
bucket_sums = grouped.contb_receipt_amt.sum().unstack(0)
normed_sums = bucket_sums.div(bucket_sums.sum(axis=1), axis=0)
normed_sums
```

In [ ]:
bucket_sums = grouped.contb_receipt_amt.sum().unstack(0)
normed_sums = bucket_sums.div(bucket_sums.sum(axis=1), axis=0)
normed_sums

```
normed_sums[:-2].plot(kind='barh')
```

In [ ]:
normed_sums[:-2].plot(kind='barh')

### Donation Statistics by State
Aggregating the data by candidate and state is a routine affair:

```
grouped = fec_mrbo.groupby(['cand_nm', 'contbr_st'])
totals = grouped.contb_receipt_amt.sum().unstack(0).fillna(0)
totals = totals[totals.sum(1) > 100000]
totals[:10]
```

In [ ]:
grouped = fec_mrbo.groupby(['cand_nm', 'contbr_st'])
totals = grouped.contb_receipt_amt.sum().unstack(0).fillna(0)
totals = totals[totals.sum(1) > 100000]
totals[:10]

If you divide each row by the total contribution amount, you get the relative percentage
of total donations by state for each candidate:

```
percent = totals.div(totals.sum(1), axis=0)
percent[:10]
```

In [ ]:
percent = totals.div(totals.sum(1), axis=0)
percent[:10]

## USA.gov Data from Bitly

In [ ]:
from numpy.random import randn
import numpy as np
np.random.seed(123)
import os
import matplotlib.pyplot as plt
import pandas as pd
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4)
pd.options.display.max_rows = 20

In 2011, URL shortening service Bitly partnered with the US government website
USA.gov to provide a feed of anonymous data gathered from users who shorten links
ending with .gov or .mil. In 2011, a live feed as well as hourly snapshots were available
as downloadable text files. This service is shut down in 2017, but here is a data file for examples.
if we read just the first line of a file we may see something like this:

In [ ]:
path = 'datasets/bitly_usagov/example.txt'
open(path).readline()

In [ ]:
import json
records = [json.loads(line) for line in open(path)]

In [ ]:
records[0]

### Counting Time Zones in Pure Python
Suppose we were interested in finding the most often-occurring time zones in the
dataset (the tz field). There are many ways we could do this. First, let’s extract a list of
time zones again using a list comprehension:

In [ ]:
time_zones = [rec['tz'] for rec in records]

Oops! Turns out that not all of the records have a time zone field. This is easy to handle,
as we can add the check if 'tz' in rec at the end of the list comprehension:

In [ ]:
time_zones = [rec['tz'] for rec in records if 'tz' in rec]
time_zones[:10]

Now, to produce counts by time zone I’ll show two approaches: the harder way (using just the Python
standard library) and the easier way (using pandas).

In [ ]:
def get_counts(sequence):
    counts = {}
    for x in sequence:
        if x in counts:
            counts[x] += 1
        else:
            counts[x] = 1
    return counts

In [ ]:
from collections import defaultdict

def get_counts2(sequence):
    counts = defaultdict(int) # values will initialize to 0
    for x in sequence:
        counts[x] += 1
    return counts

In [ ]:
counts = get_counts(time_zones)

In [ ]:
counts['America/New_York']

In [ ]:
len(time_zones)

In [ ]:
def top_counts(count_dict, n=10):
    value_key_pairs = [(count, tz) for tz, count in count_dict.items()]
    value_key_pairs.sort()
    return value_key_pairs[-n:]

In [ ]:
top_counts(counts)

If you search the Python standard library, you may find the collections.Counter
class, which makes this task a lot easier:

In [ ]:
from collections import Counter
counts = Counter(time_zones)
counts.most_common(10)

### Counting Time Zones with pandas

Creating a DataFrame from the original set of records is as easy as passing the list of
records to pandas.DataFrame:

In [ ]:
import pandas as pd
frame = pd.DataFrame(records)

In [ ]:
frame.info()

In [ ]:
frame['tz'][:10]

In [ ]:
tz_counts = frame['tz'].value_counts()
tz_counts[:10]

We can visualize this data using matplotlib. You can do a bit of munging to fill in a
substitute value for unknown and missing time zone data in the records. We replace
the missing values with the fillna method and use boolean array indexing for the
empty strings:

In [ ]:
clean_tz = frame['tz'].fillna('Missing')

In [ ]:
clean_tz[clean_tz == ''] = 'Unknown'

In [ ]:
tz_counts = clean_tz.value_counts()

In [ ]:
tz_counts[:10]

In [ ]:
plt.figure(figsize=(10, 4))

In [ ]:
import seaborn as sns
subset = tz_counts[:10]
sns.barplot(y=subset.index, x=subset.values)

The a field contains information about the browser, device, or application used to
perform the URL shortening:

In [ ]:
frame['a'][1]

In [ ]:
frame['a'][50]

In [ ]:
frame['a'][51][:50]  # long line

Parsing all of the interesting information in these “agent” strings may seem like a
daunting task. One possible strategy is to split off the first token in the string (corresponding
roughly to the browser capability) and make another summary of the user
behavior:

In [ ]:
results = pd.Series([x.split()[0] for x in frame.a.dropna()])

In [ ]:
results[:5]

In [ ]:
results.value_counts()[:8]

Now, suppose you wanted to decompose the top time zones into Windows and non-
Windows users. As a simplification, let’s say that a user is on Windows if the string
'Windows' is in the agent string. Since some of the agents are missing, we’ll exclude
these from the data:

In [ ]:
cframe = frame[frame.a.notnull()]

In [ ]:
cframe['os'] = np.where(cframe['a'].str.contains('Windows'),
                        'Windows', 'Not Windows')

In [ ]:
cframe['os'][:5]

In [ ]:
by_tz_os = cframe.groupby(['tz', 'os'])

In [ ]:
by_tz_os.size()

In [ ]:
agg_counts = by_tz_os.size().unstack().fillna(0)

In [ ]:
agg_counts[:10]

Finally, let’s select the top overall time zones. To do so, I construct an indirect index
array from the row counts in agg_counts:

In [ ]:
agg_counts.sum(1).argsort()

In [ ]:
# Use to sort in ascending order
indexer = agg_counts.sum(1).argsort()

In [ ]:
indexer[:10]

In [ ]:
count_subset = agg_counts.take(indexer[-10:])
count_subset

pandas has a convenience method called nlargest that does the same thing:

In [ ]:
agg_counts.sum(1).nlargest(10)

In [ ]:
plt.figure()

In [ ]:
# Rearrange the data for plotting
count_subset = count_subset.stack()
count_subset.name = 'total'
count_subset

In [ ]:
count_subset = count_subset.reset_index()
count_subset[:10]

In [ ]:
sns.barplot(x='total', y='tz', hue='os',  data=count_subset)

The plot doesn’t make it easy to see the relative percentage of Windows users in the
smaller groups, so let’s normalize the group percentages to sum to 1:

In [ ]:
def norm_total(group):
    group['normed_total'] = group.total / group.total.sum()
    return group

In [ ]:
results = count_subset.groupby('tz').apply(norm_total)

In [ ]:
plt.figure()

In [ ]:
sns.barplot(x='normed_total', y='tz', hue='os',  data=results)

We could have computed the normalized sum more efficiently by using the trans
form method with groupby:

In [ ]:
g = count_subset.groupby('tz')
results2 = count_subset.total / g.total.transform('sum')